# Data Cleaning Step

The purpose of this notebook is to document the steps to clean up the dataset to prepare it for future analysis. The notebook will follow the following cleaning steps:

1. coerce passenger count and ratecodeID into integer data types
2. remove those rows with questionable pick up and drop off times
3. remove the rows where the drop off time is earlier than the pick up time
4. remove the rows where the pick up time is outside of Janurary
5. mark the rows where there are negative totals
6. mark the rows where there are negative totals and there are reversal positive matching rows

In [1]:
# Imports

import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# Read in the files

df = pd.read_parquet("../data/raw/yellow_tripdata_2026-01.parquet")
taxi_zone = pd.read_csv("../data/raw/taxi_zone_lookup.csv")

In [3]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


In [4]:
taxi_zone.head()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


### Step 1: Coerce Passenger Count into Integer Data Types

In [5]:
df.dtypes.reset_index()

,index,0
0,VendorID,int32
1,tpep_pickup_datetime,datetime64[us]
2,tpep_dropoff_datetime,datetime64[us]
3,passenger_count,float64
4,trip_distance,float64
5,RatecodeID,float64
6,store_and_fwd_flag,str
7,PULocationID,int32
8,DOLocationID,int32
9,payment_type,int64


In [6]:
df['RatecodeID'] = df['RatecodeID'].astype('Int32')
df[['RatecodeID']].sample(10)

,RatecodeID
2176660,1
1937983,99
3600558,<NA>
2801741,<NA>
3681075,<NA>
685833,1
1659996,1
2903777,<NA>
3646595,<NA>
1308878,1


In [7]:
df['passenger_count'] = df['passenger_count'].astype('Int32')
df[['passenger_count']].sample(10)

,passenger_count
1121529,1
2009544,2
2032616,1
3336105,<NA>
3025543,<NA>
727533,1
2213888,1
1225278,1
774277,2
3374319,<NA>


### Step 2: Remove columns with questionable pick up times (dropff time that is earlier thatn pick up time)

In [8]:
df[df['tpep_pickup_datetime'] > df['tpep_dropoff_datetime']]

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
1465434,1,2026-01-17 19:30:00,2026-01-17 19:18:18,1,16.5,99,N,46,86,1,66.5,0.0,0.5,0.0,0.0,0.0,67.0,0.0,0.0,0.0


In [9]:
df.drop(index=1465434, inplace=True)

In [10]:
df[df['tpep_pickup_datetime'] > df['tpep_dropoff_datetime']]

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee


this row has been removed

## Step 3: Remove the rows where the pick up time is earlier than Jaurary

In [11]:
start_date = pd.Timestamp("2026-01-01")
outside_bound_list = df[df['tpep_pickup_datetime'] < start_date].index.tolist()
outside_bound_list

[980, 981, 982, 1404, 2289, 5181]

In [12]:
df.drop(index=outside_bound_list, inplace=True)

In [13]:
df[df['tpep_dropoff_datetime'] < start_date]

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee


In [14]:
end_date = pd.Timestamp('2026-02-01')
outside_bound_list = df[df['tpep_pickup_datetime'] >= end_date].index.tolist()
outside_bound_list

[2635466]

In [15]:
df[df['tpep_pickup_datetime'] >= end_date]

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
2635466,2,2026-02-01 00:45:01,2026-02-01 01:03:08,1,1.78,1,N,79,249,1,15.6,1.0,0.5,5.34,0.0,1.0,26.69,2.5,0.0,0.75


In [16]:
df.drop(index=outside_bound_list, inplace=True)

In [17]:
df[df['tpep_pickup_datetime'] >= end_date]

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee


pick up times that are later than end date are also dropped

### Step 4: Mark the rows where the totals are negative and also if there are reversal transactions

In [18]:
df['is_negative_total'] = df['total_amount'] < 0
df[['is_negative_total']].sample(10)

,is_negative_total
700997,False
1833707,False
222724,False
2509992,False
968441,False
1391215,False
2742695,False
3321479,False
84995,False
2453622,False


In [19]:
df['is_matched_reversal'] = False

In [20]:
match_cols = [
    "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "trip_distance"
]

negative_trips = df[df['is_negative_total'] == True]
positive_trips = df[df['is_negative_total'] == False]

matched_pairs = negative_trips.merge(positive_trips, on=match_cols, suffixes=("_neg", "_pos"))
matched_pairs.shape

(35712, 39)

In [21]:
matched_pairs.sample(10)

,VendorID_neg,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count_neg,trip_distance,RatecodeID_neg,store_and_fwd_flag_neg,PULocationID,DOLocationID,payment_type_neg,...,mta_tax_pos,tip_amount_pos,tolls_amount_pos,improvement_surcharge_pos,total_amount_pos,congestion_surcharge_pos,Airport_fee_pos,cbd_congestion_fee_pos,is_negative_total_pos,is_matched_reversal_pos
32009,2,2026-01-29 02:42:08,2026-01-29 02:46:53,1,0.97,1,N,263,75,4,...,0.5,0.0,0.0,1.0,12.20,2.5,0.00,0.00,False,False
16777,2,2026-01-15 09:08:47,2026-01-15 09:10:52,1,0.00,2,N,132,132,2,...,0.5,0.0,0.0,1.0,73.25,0.0,1.75,0.00,False,False
4011,2,2026-01-04 02:08:28,2026-01-04 02:10:31,1,0.00,1,N,114,114,4,...,0.5,0.0,0.0,1.0,9.45,2.5,0.00,0.75,False,False
9977,2,2026-01-09 13:21:43,2026-01-09 13:29:05,2,1.44,1,N,229,137,4,...,0.5,0.0,0.0,1.0,14.05,2.5,0.00,0.75,False,False
1202,2,2026-01-01 22:13:00,2026-01-01 22:27:37,4,2.74,1,N,238,48,4,...,0.5,0.0,0.0,1.0,22.75,2.5,0.00,0.75,False,False
16146,2,2026-01-14 17:26:02,2026-01-14 17:29:06,1,0.49,1,N,143,142,2,...,0.5,0.0,0.0,1.0,11.60,2.5,0.00,0.00,False,False
12349,2,2026-01-11 10:16:18,2026-01-11 10:30:33,3,3.53,1,N,163,74,4,...,0.5,0.0,0.0,1.0,23.15,2.5,0.00,0.75,False,False
15671,2,2026-01-14 11:15:01,2026-01-14 11:28:39,4,1.16,1,N,211,232,4,...,0.5,0.0,0.0,1.0,16.85,2.5,0.00,0.75,False,False
24086,2,2026-01-21 12:40:15,2026-01-21 12:41:52,1,0.26,1,N,239,239,4,...,0.5,0.0,0.0,1.0,7.70,2.5,0.00,0.00,False,False
9759,2,2026-01-09 09:54:06,2026-01-09 09:59:47,1,0.68,1,N,75,236,4,...,0.5,0.0,0.0,1.0,11.20,2.5,0.00,0.00,False,False


In [22]:
df["reversal_key"] = list(
    zip(
        df["tpep_pickup_datetime"],
        df["tpep_dropoff_datetime"],
        df["PULocationID"],
        df["DOLocationID"],
        df["trip_distance"],
        df["total_amount"]
    )
)

matched_negative_keys = set(
    zip(
        matched_pairs["tpep_pickup_datetime"],
        matched_pairs["tpep_dropoff_datetime"],
        matched_pairs["PULocationID"],
        matched_pairs["DOLocationID"],
        matched_pairs["trip_distance"],
        matched_pairs["total_amount_neg"]
    )
)

df['is_matched_reversal'] = df['reversal_key'].isin(matched_negative_keys)

In [23]:
df.sample(10)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,is_negative_total,is_matched_reversal,reversal_key
1730615,2,2026-01-21 06:50:14,2026-01-21 06:57:49,1,3.35,1,N,263,233,1,...,2.00,0.0,1.0,21.60,2.5,0.0,0.00,False,False,"(2026-01-21 06:50:14, 2026-01-21 06:57:49, 263..."
350818,2,2026-01-05 18:02:21,2026-01-05 18:10:37,1,1.15,1,N,163,100,2,...,0.00,0.0,1.0,16.55,2.5,0.0,0.75,False,False,"(2026-01-05 18:02:21, 2026-01-05 18:10:37, 163..."
870389,2,2026-01-11 14:48:54,2026-01-11 14:58:44,1,1.46,1,N,42,42,2,...,0.00,0.0,1.0,12.90,0.0,0.0,0.00,False,False,"(2026-01-11 14:48:54, 2026-01-11 14:58:44, 42,..."
15406,2,2026-01-01 03:13:24,2026-01-01 03:39:37,1,5.57,1,N,237,255,1,...,6.78,0.0,1.0,40.68,2.5,0.0,0.75,False,False,"(2026-01-01 03:13:24, 2026-01-01 03:39:37, 237..."
2339462,2,2026-01-28 20:10:43,2026-01-28 20:22:13,1,0.93,1,N,162,142,1,...,3.43,0.0,1.0,20.58,2.5,0.0,0.75,False,False,"(2026-01-28 20:10:43, 2026-01-28 20:22:13, 162..."
3093184,2,2026-01-16 10:42:58,2026-01-16 11:10:25,<NA>,4.69,<NA>,NaN,237,42,0,...,0.00,0.0,1.0,39.97,NaN,NaN,0.75,False,False,"(2026-01-16 10:42:58, 2026-01-16 11:10:25, 237..."
3704981,2,2026-01-31 18:34:47,2026-01-31 18:46:58,<NA>,2.73,<NA>,NaN,140,75,0,...,0.00,0.0,1.0,25.74,NaN,NaN,0.00,False,False,"(2026-01-31 18:34:47, 2026-01-31 18:46:58, 140..."
3454225,2,2026-01-26 02:52:50,2026-01-26 03:06:57,<NA>,7.15,<NA>,NaN,263,88,0,...,0.00,0.0,1.0,49.20,NaN,NaN,0.75,False,False,"(2026-01-26 02:52:50, 2026-01-26 03:06:57, 263..."
2215574,2,2026-01-27 14:19:18,2026-01-27 14:29:39,1,1.22,1,N,238,236,1,...,2.94,0.0,1.0,17.64,2.5,0.0,0.00,False,False,"(2026-01-27 14:19:18, 2026-01-27 14:29:39, 238..."
1407293,1,2026-01-17 03:11:29,2026-01-17 03:22:21,1,1.70,1,N,148,137,1,...,3.55,0.0,1.0,21.40,2.5,0.0,0.75,False,False,"(2026-01-17 03:11:29, 2026-01-17 03:22:21, 148..."


In [24]:
df.drop(columns=['reversal_key'], inplace=True)

In [25]:
df.sample(10)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,is_negative_total,is_matched_reversal
101221,2,2026-01-02 15:35:04,2026-01-02 15:54:25,1,3.65,1,N,246,237,1,...,0.5,7.37,0.0,1.0,31.92,2.5,0.00,0.75,False,False
2510666,1,2026-01-30 17:46:13,2026-01-30 18:51:11,1,6.70,99,N,36,163,1,...,0.5,0.00,0.0,0.0,32.00,0.0,0.00,0.00,False,False
2277220,1,2026-01-28 09:39:27,2026-01-28 10:03:15,3,2.20,1,Y,237,186,1,...,0.5,5.05,0.0,1.0,30.30,2.5,0.00,0.75,False,False
1320426,2,2026-01-16 10:33:46,2026-01-16 10:55:59,5,2.76,1,N,140,238,1,...,0.5,2.50,0.0,1.0,27.70,2.5,0.00,0.00,False,False
2657788,2,2026-01-01 06:43:19,2026-01-01 06:54:52,<NA>,2.62,<NA>,NaN,41,141,0,...,0.5,0.00,0.0,1.0,33.96,NaN,NaN,0.00,False,False
677106,2,2026-01-09 14:40:51,2026-01-09 14:47:30,1,1.17,1,N,138,70,1,...,0.5,3.60,0.0,1.0,19.75,0.0,1.75,0.00,False,False
1839080,2,2026-01-22 09:53:38,2026-01-22 10:18:44,2,5.62,1,N,186,87,1,...,0.5,7.29,0.0,1.0,43.74,2.5,0.00,0.75,False,False
2505897,2,2026-01-30 17:27:01,2026-01-30 17:31:07,1,0.53,1,N,90,90,1,...,0.5,2.61,0.0,1.0,15.66,2.5,0.00,0.75,False,False
3681834,2,2026-01-31 05:46:32,2026-01-31 05:58:23,<NA>,3.27,<NA>,NaN,129,223,0,...,0.5,0.00,0.0,1.0,23.73,NaN,NaN,0.00,False,False
3079086,2,2026-01-15 22:36:24,2026-01-15 22:53:22,<NA>,3.85,<NA>,NaN,79,141,0,...,0.5,0.00,0.0,1.0,30.11,NaN,NaN,0.75,False,False


In [26]:
df.columns.tolist()

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee',
 'is_negative_total',
 'is_matched_reversal']

In [29]:
df.isna().sum()

VendorID                       0
tpep_pickup_datetime           0
tpep_dropoff_datetime          0
passenger_count          1088058
trip_distance                  0
RatecodeID               1088058
store_and_fwd_flag       1088058
PULocationID                   0
DOLocationID                   0
payment_type                   0
fare_amount                    0
extra                          0
mta_tax                        0
tip_amount                     0
tolls_amount                   0
improvement_surcharge          0
total_amount                   0
congestion_surcharge     1088058
Airport_fee              1088058
cbd_congestion_fee             0
is_negative_total              0
is_matched_reversal            0
dtype: int64

In [28]:
df.to_csv("../data/raw/yellow_tripdata_2026-01_cleaned.csv", index=False)